# Pipeline de Ingesta de Datos — Entrega 1

**Proyecto Integrador — Ciencia de Datos (UTN FRM 2026)**

**Tema:** degradación de neumáticos en Fórmula 1 ("the cliff").

Este notebook arma el **pipeline automatizado** que produce el dataset:

1. Descarga vuelta-a-vuelta desde los servidores oficiales de la F1 usando **FastF1**.
2. Mergea el clima a cada vuelta.
3. Deriva variables (corrección por combustible) y flags de limpieza.
4. Filtra a vueltas representativas.
5. Verifica la integridad del dataset resultante.

Cada fila del dataset final es **una vuelta de un piloto en una carrera** (datos tidy).

**Fuente:** [FastF1](https://docs.fastf1.dev/) — consulta directa a la telemetría/cronometraje oficial de la F1. Sin descarga manual de CSVs.

## Sección 0 — Configuración

Importamos FastF1 y activamos la **caché en disco**: la telemetría descargada se guarda en `cache/`, así la primera corrida baja de los servidores (lento) y las siguientes leen de disco (instantáneo).

In [ ]:
import warnings
from pathlib import Path

import fastf1
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

BASE_DIR = Path.cwd()
CACHE_DIR = BASE_DIR / "cache"
DATA_DIR = BASE_DIR / "data"
CACHE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))
print(f"FastF1 {fastf1.__version__} | cache: {CACHE_DIR}")

## Sección 1 — Parámetros del pipeline

`QUICK_TEST = True` baja solo unas pocas carreras para que el notebook corra de punta a punta en minutos (**Restart & Run All**). Para generar el **dataset completo** poné `QUICK_TEST = False`: **una temporada (~21.000 vueltas limpias)** ya supera con creces el ideal de >10.000 filas. La primera corrida puede tardar varios minutos por temporada; después la caché la hace instantánea. Si querés más variedad de circuitos/temperaturas, agregá años en `SEASONS`.

**Sobre el combustible:** la F1 no publica la carga de combustible, así que **no la estimamos ni fabricamos ninguna columna**. Su efecto (el auto se aliviana y va más rápido vuelta a vuelta) queda representado por `LapNumber` (la vuelta de carrera, dato real medido). Ver la justificación en el `README.md`.

In [ ]:
QUICK_TEST = True                 # True = prueba rápida | False = dataset completo
# Una temporada (~21.000 vueltas limpias) ya supera el ideal de >10.000 filas.
# Sumá años (ej. [2023, 2024, 2025]) solo si querés más variedad de circuitos/temperaturas.
SEASONS = [2024] if QUICK_TEST else [2024]
MAX_RACES = 3 if QUICK_TEST else None

# NOTA: no estimamos el combustible; su efecto queda representado por LapNumber (dato real).

LAP_COLS = [
    "Driver", "DriverNumber", "Team", "LapNumber", "Stint",
    "LapTime", "Sector1Time", "Sector2Time", "Sector3Time",
    "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
    "Compound", "TyreLife", "FreshTyre",
    "PitInTime", "PitOutTime", "TrackStatus", "Position",
    "Deleted", "IsAccurate", "LapStartTime",
]
WEATHER_COLS = ["AirTemp", "TrackTemp", "Humidity", "Pressure",
                "WindSpeed", "WindDirection", "Rainfall"]
print(f"Temporadas: {SEASONS} | max carreras: {MAX_RACES}")

## Sección 2 — Función de ingesta por carrera

Para cada carrera cargamos las vueltas (`session.laps`), mergeamos el clima alineado a cada vuelta, convertimos los tiempos (timedelta) a segundos, y agregamos flags de limpieza y la corrección por combustible. Usamos `telemetry=False` porque no necesitamos la telemetría de canal completo (pesa mucho); sí cargamos el clima.

In [ ]:
def process_session(year, rnd, event_name):
    """Carga una carrera y devuelve sus vueltas como DataFrame tidy (o None si falla)."""
    try:
        session = fastf1.get_session(year, rnd, "R")
        session.load(telemetry=False, weather=True, messages=False)
    except Exception as exc:
        print(f"  [skip] {year} R{rnd} {event_name}: {exc}")
        return None

    laps = session.laps
    if laps is None or len(laps) == 0:
        print(f"  [skip] {year} R{rnd} {event_name}: sin vueltas")
        return None

    df = laps[[c for c in LAP_COLS if c in laps.columns]].copy()

    # Clima alineado a cada vuelta
    try:
        weather = laps.get_weather_data().reset_index(drop=True)
        for c in WEATHER_COLS:
            if c in weather.columns:
                df[c] = weather[c].values
    except Exception as exc:
        print(f"  [warn] {year} R{rnd}: sin clima ({exc})")

    # timedelta -> segundos
    for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
        if col in df.columns:
            df[col] = df[col].dt.total_seconds()

    # Flags de limpieza
    df["IsPitLap"] = df["PitInTime"].notna() | df["PitOutTime"].notna()
    df["IsGreen"] = df["TrackStatus"].astype(str) == "1"   # 1 = pista verde

    # Metadata
    df.insert(0, "Year", year)
    df.insert(1, "Round", rnd)
    df.insert(2, "Event", event_name)

    print(f"  [ok] {year} R{rnd} {event_name}: {len(df)} vueltas")
    return df

## Sección 3 — Descarga de las temporadas

Recorremos el calendario de cada temporada (`get_event_schedule`) y descargamos carrera por carrera. Si `QUICK_TEST=True`, solo las primeras `MAX_RACES`.

In [ ]:
all_laps = []
for year in SEASONS:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rounds = schedule["RoundNumber"].tolist()
    if MAX_RACES:
        rounds = rounds[:MAX_RACES]
    print(f"\n=== Temporada {year}: {len(rounds)} carreras ===")
    for rnd in rounds:
        event_name = schedule.loc[schedule["RoundNumber"] == rnd, "EventName"].iloc[0]
        df = process_session(year, rnd, event_name)
        if df is not None:
            all_laps.append(df)

raw = pd.concat(all_laps, ignore_index=True)
print(f"\nVueltas totales descargadas: {len(raw):,} | columnas: {raw.shape[1]}")

## Sección 4 — Limpieza y selección de columnas

El tiempo de vuelta se **contamina** con Safety Car / VSC, banderas amarillas, tráfico, vueltas de entrada/salida de boxes y vueltas borradas por límites de pista. Si no filtramos, la señal de degradación se ahoga. Nos quedamos con vueltas **verdes y representativas** usando los flags de FastF1.

Además nos quedamos solo con las **columnas que vamos a usar** (`LEAN_COLS`): identificación mínima + features + target. Descartamos sectores y velocidades (son partes del `LapTime` → *leakage*), `DriverNumber` (redundante con `Driver`) y las columnas que solo servían para filtrar. El dataset completo con todas las columnas queda igual en `f1_laps_raw.csv`.

In [ ]:
# Columnas del dataset "lean" (listo para modelar): identificación mínima + features + target.
# Dejamos afuera sectores/velocidades (son partes del LapTime -> leakage), DriverNumber
# (redundante con Driver) y las columnas que solo sirven para filtrar.
LEAN_COLS = [
    "Year", "Round", "Event", "Driver",                         # identificación / contexto
    "LapNumber", "Stint", "TyreLife", "Compound", "FreshTyre",   # features
    "TrackTemp", "AirTemp", "Humidity", "Rainfall",             # features (clima)
    "LapTime",                                                   # TARGET
]

def clean(df):
    mask = (
        df["IsAccurate"].fillna(False)     # FastF1 marca la vuelta como fiable
        & df["IsGreen"]                    # pista verde (sin SC/VSC/amarilla)
        & ~df["IsPitLap"]                  # sin entrada/salida de boxes
        & ~df["Deleted"].fillna(False)     # no borrada por límites de pista
        & df["LapTime"].notna()
        & df["Compound"].notna()
    )
    filtered = df[mask]
    return filtered[[c for c in LEAN_COLS if c in filtered.columns]].copy()

cleaned = clean(raw)
print(f"Vueltas limpias: {len(cleaned):,} × {cleaned.shape[1]} columnas "
      f"({len(cleaned)/len(raw)*100:.1f}% del total)")

## Sección 5 — Guardado

In [ ]:
raw.to_csv(DATA_DIR / "f1_laps_raw.csv", index=False)
cleaned.to_csv(DATA_DIR / "f1_laps_clean.csv", index=False)
print("Guardado:")
print(f"  data/f1_laps_raw.csv   -> {len(raw):,} vueltas")
print(f"  data/f1_laps_clean.csv -> {len(cleaned):,} vueltas")

## Sección 6 — Verificación

Chequeos de integridad y una visualización de la curva de degradación.

In [ ]:
print(f"Filas (vueltas)   : {len(cleaned):,}")
print(f"Columnas          : {cleaned.shape[1]}")
print(f"Carreras          : {cleaned.groupby(['Year','Round']).ngroups}")
print(f"Pilotos únicos    : {cleaned['Driver'].nunique()}")
print(f"Compuestos        : {sorted(cleaned['Compound'].dropna().unique())}")
print(f"Nulos en LapTime  : {cleaned['LapTime'].isna().sum()}")
print()
cleaned[['Year','Round','Driver','LapNumber','Stint','Compound','TyreLife','LapTime','TrackTemp']].head(8)

### Curva de degradación (validación visual)

Tiempo de vuelta vs. vida del neumático, en un stint de ejemplo. Acá es donde debería verse el "cliff": una zona casi plana que en cierto punto se dispara.

In [ ]:
# Un stint largo de ejemplo (el que más vueltas tiene en el dataset)
grp = cleaned.groupby(['Year','Round','Driver','Stint'])
key = grp.size().idxmax()
stint = grp.get_group(key).sort_values('TyreLife')
compound = stint['Compound'].iloc[0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(stint['TyreLife'], stint['LapTime'], marker='o')
ax.set_xlabel('Vida del neumático (vueltas)')
ax.set_ylabel('Tiempo de vuelta (s)')
ax.set_title(f"Degradación — {key[2]}, stint {int(key[3])} ({compound}) | {key[0]} R{key[1]}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()